# Análisis univariado

In [1]:
from eda_utils import *   # carga y prepara la base (capítulo 1)

In [2]:
p = viv.precio_M
fig = make_subplots(1, 2, subplot_titles=("Precio (hasta el percentil 99)", "Precio en escala logarítmica"))
fig.add_histogram(x=p[p <= p.quantile(0.99)], nbinsx=60, marker_color=PRINCIPAL, row=1, col=1,
                  hovertemplate="%{x} M: %{y} ventas<extra></extra>")
fig.add_vline(x=p.median(), line_color=OSCURO, line_width=2, row=1, col=1,
              annotation_text=f"mediana {p.median():,.0f} M")
fig.add_histogram(x=np.log10(viv.precio_real), nbinsx=60, marker_color=PRINCIPAL, row=1, col=2,
                  hovertemplate="%{y} ventas<extra></extra>")
marcas = [1e7, 3e7, 1e8, 3e8, 1e9, 3e9]
fig.update_xaxes(tickvals=np.log10(marcas), ticktext=["10 M", "30 M", "100 M", "300 M", "1.000 M", "3.000 M"], row=1, col=2)
fig.update_xaxes(title_text="millones de pesos de 2025", row=1, col=1)
fig.update_layout(showlegend=False, bargap=0.03)
mostrar(fig, "Distribución del precio de venta")

viv[["precio_real", "area", "precio_m2_real"]].describe(percentiles=[.05, .25, .5, .75, .95]).T

,count,mean,std,min,5%,25%,50%,75%,95%,max
precio_real,"34,606.00","275,488,577.42","338,704,963.96","14,944,446.34","65,881,526.70","128,463,915.58","194,784,753.00","335,163,712.88","705,116,962.34","32,022,433,813.42"
area,"34,606.00",88.62,294.88,5.00,41.00,50.00,64.00,95.00,199.00,"52,168.00"
precio_m2_real,"34,606.00","3,318,797.12","1,947,606.01","310,661.49","849,593.53","2,112,177.64","2,994,953.62","4,366,489.28","6,273,484.61","36,130,848.76"


La vivienda mediana cuesta 195 M de 2025 y tiene 64 m². El precio tiene una cola
larga a la derecha que desaparece en escala logarítmica.

In [3]:
fig = make_subplots(1, 2, subplot_titles=("Precio", "log10(precio)"))
for i, col in enumerate(["precio_real", "log_precio"], 1):
    (teo, obs), (pend, corte, _) = stats.probplot(viv[col], dist="norm")
    fig.add_scatter(x=teo, y=obs, mode="markers", marker=dict(size=3, color=PRINCIPAL, opacity=0.4),
                    row=1, col=i, hoverinfo="skip")
    fig.add_scatter(x=teo, y=pend * teo + corte, mode="lines", line=dict(color=OSCURO), row=1, col=i)
fig.update_xaxes(title_text="cuantiles de una normal")
fig.update_layout(showlegend=False)
mostrar(fig, "Gráficos Q-Q: qué tan cerca de la normal está el precio", alto=400)

En pesos el precio se aleja por completo de la normal; en logaritmo sigue la recta salvo
en los extremos (asimetría de 0,23). Por eso el modelo se entrenará sobre log(precio).

In [4]:
fig = make_subplots(1, 4, subplot_titles=("Habitaciones", "Baños", "Piso", "Edad (años)"))
for i, (col, tope) in enumerate([("habitaciones", 7), ("banios", 6), ("piso", 20)], 1):
    vc = viv[col].dropna().round().astype(int).clip(upper=tope).value_counts().sort_index()
    etiquetas = [str(v) if v < tope else f"{tope}+" for v in vc.index]
    fig.add_bar(x=etiquetas, y=vc.values, marker_color=PRINCIPAL, row=1, col=i,
                hovertemplate="%{x}: %{y} ventas<extra></extra>")
fig.add_histogram(x=viv.edad, nbinsx=40, marker_color=PRINCIPAL, row=1, col=4,
                  hovertemplate="%{x} años: %{y} ventas<extra></extra>")
fig.update_layout(showlegend=False, bargap=0.1)
mostrar(fig, "Características de las viviendas", alto=380)

viv[["habitaciones", "banios", "piso", "anio_construccion", "edad"]].describe(percentiles=[.05, .5, .95]).T

,count,mean,std,min,5%,50%,95%,max
habitaciones,"34,595.00",2.77,0.74,0.00,2.00,3.00,4.00,14.00
banios,"34,604.00",1.78,0.78,0.00,1.00,2.00,3.00,20.00
piso,"30,616.00",4.21,3.75,1.00,1.00,3.00,12.00,31.00
anio_construccion,"34,602.00","2,004.46",20.70,"1,932.00","1,952.00","2,014.00","2,022.00","2,025.00"
edad,"33,362.00",18.25,20.73,0.00,1.00,9.00,69.00,91.00


La vivienda típica tiene 3 habitaciones, 2 baños y 9 años de antigüedad. La mayoría
está en pisos bajos.

In [5]:
fig = make_subplots(1, 2, subplot_titles=("Estrato", "Tipo de predio"), horizontal_spacing=0.15)
e = viv.estrato_txt.value_counts().reindex(NOMBRES_ESTRATO + ["Sin estrato"]).dropna()
fig.add_bar(x=e.index.str.replace("Estrato ", ""), y=e.values, row=1, col=1,
            marker_color=COLORES_ESTRATO + ["#DDDDDD"], hovertemplate="%{x}: %{y} ventas<extra></extra>")
c = viv.condicion_predio.value_counts().sort_values()
fig.add_bar(x=c.values, y=c.index, orientation="h", row=1, col=2, marker_color=OSCURO,
            hovertemplate="%{y}: %{x} ventas<extra></extra>")
fig.update_layout(showlegend=False)
mostrar(fig, "Estrato y tipo de predio", alto=380)

Los estratos 1 a 4 suman el 84% de las ventas, y el 84% son unidades en propiedad
horizontal, es decir, apartamentos. En la versión anterior eran el 60%: las ventas que entraron
con el cruce por matrícula del IGAC son casi todas apartamentos.